In [ ]:
import json
import requests
from time import sleep
from numpy.random import randint
from os import environ
import pygsheets
from utils import read_chunks

## Solicitando contato do Proprietário

In [ ]:
codigo = '00011'

lista_proprietarios = read_chunks(
  f'{codigo} - proprietarios_anuncios_*.json', fr'./dados/{codigo}'
)

In [ ]:
len(lista_proprietarios)

### Tratando dado extraído para deixar no formato da requisição

In [ ]:
headers = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "authorization": '97c59f98e2f14ae289f952460882559b',
    "content-type": "application/json",
    "priority": "u=1, i",
    "sec-ch-ua": "\"Google Chrome\";v=\"125\", \"Chromium\";v=\"125\", \"Not.A/Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

In [ ]:
with open('dados/proprietarios_consultados.json') as f:
    proprietarios_consultados = set(json.loads(f.read()))

In [ ]:
with open('dados/resultados_consultados.json') as f:
    lista_contatos = json.loads(f.read())

In [ ]:
infos_necessarias =[
    'property_type',
    'endereco_completo',
    'district',
    'name',
    'phones',
    'buy_price',
    'total_area',
]

In [ ]:
def tratar_dados_para_request_contato(anuncio: dict) -> dict:
    formato_final = {}
    formato_final['announce_id'] = str(anuncio['announce_id'])

    if 'birthDate' in anuncio['data']['people'][0]:
        del anuncio['data']['people'][0]['birthDate']

    anuncio['data']['people'][0]['document_type'] = anuncio['data']['people'][0]['type']
    anuncio['data']['people'][0]['type'] = 'DOCUMENT'

    formato_final['search'] = {}
    formato_final['search']['key'] = anuncio['data']['people'][0]

    return formato_final

In [ ]:
lista_proprietarios[0]

In [ ]:
lista_proprietarios_filtrados = []
lista_documentos_filtrados = []

for proprietario in lista_proprietarios:
    if 'data' not in proprietario:
        continue

    if proprietario['data']['totalElements'] != 1:
        print('Mais de um proprietário possível, passando para o próximo.')
        continue
    
    if 'document' not in proprietario['data']['people'][0]:
        continue

    documento_proprietario = str(proprietario['data']['people'][0]['document'])

    if len(documento_proprietario) != 11:
        print(f'{documento_proprietario} não é cpf, passando para o próximo.')
        continue

    if documento_proprietario in lista_documentos_filtrados:
        print(f'Documento: {documento_proprietario} já está na lista.')
        continue

    lista_documentos_filtrados.append(documento_proprietario)
    lista_proprietarios_filtrados.append(proprietario)

In [ ]:
docs = []

for proprietario in lista_proprietarios_filtrados:
    documento_proprietario = str(proprietario['data']['people'][0]['document'])
    docs.append(documento_proprietario)

In [ ]:
docs[:10]

In [ ]:
len(lista_proprietarios_filtrados)

In [ ]:
proprietarios_processados = []

In [ ]:
limite = 242

for index, proprietario in enumerate(lista_proprietarios_filtrados):
    if len(proprietarios_processados) >= limite:
        break

    print(f'Item: {index}')

    try:
        documento_proprietario = str(proprietario['data']['people'][0]['document'])

        if documento_proprietario in proprietarios_consultados:
            print(f'Documento: {documento_proprietario} já pesquisado.')
            proprietarios_processados.append(proprietario)
            continue

        print(f'Consultando: {documento_proprietario}')

        body_contato = tratar_dados_para_request_contato(proprietario)

        response = requests.post(
            url="https://painel.fisgar.com.br/api123/v1/api/region/sp/owners/GetPersonInfo",
            headers = headers,
            json = body_contato
        )

        if response.status_code != 200:
            print(f'Erro na requisição: {response.content.decode()}')
            continue

        proprietarios_consultados.add(documento_proprietario)

        if 'data' not in response.json():
            print(f'Erro na requisição: {response.json()}')
            continue

        resultado_json = response.json()['data']
        dados_proprietario = resultado_json['people'][0]
        dados_proprietario['searchId'] = resultado_json['searchId']
        dados_proprietario['announce_id'] = body_contato['announce_id']

        lista_contatos.append(dados_proprietario)
        proprietarios_processados.append(proprietario)

        sleep(randint(1,4))

    except Exception as e:
        print(f'Erro inesperado: {e}')


In [ ]:
len(lista_contatos)

In [ ]:
with open('dados/proprietarios_consultados.json', 'w') as f:
    f.write(json.dumps(list(proprietarios_consultados)))

In [ ]:
with open('dados/resultados_consultados.json', 'w') as f:
    f.write(json.dumps(lista_contatos))

## Tratar dados os contatos

In [ ]:
import pandas as pd

In [ ]:
with open('dados/resultados_consultados.json') as f:
    lista_contatos = json.loads(f.read())

In [ ]:
def ordenar_lista_de_dicionarios(lista: dict, chave_ordenacao: str, reverse: bool = False):
    niveis_filtro = [10, 2, 1]
    for nivel in niveis_filtro:
        lista_filtrada = sorted(list(filter(lambda x: x['score'] == nivel, lista)), key = lambda x: x[chave_ordenacao], reverse = reverse)

        if len(lista_filtrada) > 0:
            return lista_filtrada



In [ ]:
for contato in lista_contatos:
    contato['whatsapp'] = ordenar_lista_de_dicionarios(
        contato['phones'],
        'score',
        True
        )
    
    contato['telefones'] = [numero for numero in contato['phones'] if numero['type'] == 'LANDLINE']

    contato['emails'] = ordenar_lista_de_dicionarios(
        contato['emails'],
        'score',
        True
        )
    
    try:
        contato['addresses'] = [
            f"{imovel.get('street', '')} {imovel.get('number', '')}, {imovel.get('extra', '')}, {imovel.get('district', '')}, {imovel.get('city', '')} / {imovel.get('state', '')}. CEP {imovel.get('zipcode', '')}"
            for imovel in contato['addresses']
        ]
    except:
        pass



In [ ]:
anuncios = read_chunks(
  f'{codigo} - anuncios*.json', fr'./dados/{codigo}'
)

In [ ]:
dicionario_anuncios = {anuncio['id'] : anuncio for anuncio in anuncios}

In [ ]:
dicionario_contatos = {contato['document']:contato for contato in lista_contatos}

In [ ]:
anuncios_processados = []

infos_necessarias =[
    'property_type',
    'endereco_completo',
    'district',
    'name',
    'whatsapp',
    'telefones',
    'buy_price',
    'total_area',
]

for proprietario in lista_proprietarios_filtrados:
    documento_proprietario = proprietario['data']['people'][0]['document']

    id_anuncio = proprietario['announce_id']
    anuncio = dicionario_anuncios[id_anuncio]

    if documento_proprietario not in dicionario_contatos:
        continue

    contato = dicionario_contatos[documento_proprietario]
    anuncio = {**anuncio, **contato}

    logradouro_complemento = f"{anuncio.get('street', '')} {anuncio.get('number', '')}, {anuncio.get('extra', '')}" if anuncio.get('extra', '') else f"{anuncio.get('street', '')} {anuncio.get('number', '')}"

    endereco_completo = f"{logradouro_complemento}, {anuncio.get('district', '')}, {anuncio.get('city', '')} / SP. CEP {anuncio.get('zipcode', '')}"
    anuncio['endereco_completo'] = endereco_completo

    anuncio['whatsapp'] = [phone['phone'] for phone in anuncio['whatsapp']]
    anuncio['telefones'] = [phone['phone'] for phone in anuncio['telefones']]

    info_necessaria = [anuncio[info] for info in infos_necessarias]

    anuncios_processados.append(info_necessaria)


In [ ]:
df_out = pd.DataFrame(anuncios_processados, columns=infos_necessarias)

In [ ]:
df_out = df_out.drop_duplicates(['endereco_completo', 'name'])

In [ ]:
len(df_out)

In [ ]:
df_out.head()

In [ ]:
df_out['whatsapp'] = df_out['whatsapp'].apply(lambda x: ', '.join(x))

In [ ]:
df_out['telefones'] = df_out['telefones'].apply(lambda x: ', '.join(x))

In [ ]:
df_out['Nulo'] = ''

In [ ]:
df_out['Origem'] = 'Fisgar'

In [ ]:
columns = df_out.columns[:-2]
columns = columns.insert(1, 'Origem')
columns = columns.insert(5, 'Nulo')
df_out = df_out[columns]

In [ ]:
df_out.to_csv(f'dados/{codigo}/output.csv', index=False)

In [ ]:
client = pygsheets.authorize(service_account_file='./dados/gcp-project-365616-6b0f83cca4c2.json')

ss = client.open_by_url('https://docs.google.com/spreadsheets/d/1H-otjPW382f7cUNvaC_p5BJCMw7nmlmuhu2JiNBx_Y0/edit?gid=195125089#gid=195125089')

ws = ss.worksheet_by_title('Importação Fisgar')

In [ ]:
ws.set_dataframe(
    df_out,
    'A1'
)